# FloodResQ AI - full pipeline (Colab), version 0.2.3
IIT Mandi Multimodal AI Hackathon 2026 - Track B: Mapping Flood Damage from Space. **Educational prototype, not an operational tool.**

**How to run:** click each cell in order, top to bottom, and wait for the green tick before the next one. Read the printed output. Do not skip cells. If a cell fails, copy the last lines of the red error and ask for help.

**Honesty rules built into this notebook**
- Every image, layer and number comes from downloads you make with your own account. Nothing is simulated or pre-filled.
- If the evidence is weak, the result says **inconclusive**. The notebook never forces a flood result.
- Published damage maps (Copernicus EMS EMSR927, UNOSAT) and post-event OpenStreetMap edits are **never loaded** until the last evaluation cell, only to score the finished and frozen result.
- 'Potentially affected' never means 'confirmed damaged'.
- The code is tested on synthetic data only (see tests/). The live services (Copernicus, Overpass, DEM download) were **not** tested by the author, so expect to debug a little.

## 1. Install and load the helper library

In [ ]:
!pip -q install rasterio geopandas folium networkx scipy shapely requests pandas matplotlib

In [ ]:
import os, sys
EXPECTED = "0.2.3"

def _load():
    sys.modules.pop("floodresq", None)
    import floodresq as m
    return m

fr = _load() if os.path.exists("floodresq.py") else None
if fr is None or getattr(fr, "__version__", None) != EXPECTED:
    if os.path.exists("floodresq.py"):
        os.remove("floodresq.py")            # remove an old copy so it can never be used by mistake
    from google.colab import files
    print("Choose the NEW floodresq.py (version %s) from your computer and wait for 100%% done..." % EXPECTED)
    files.upload()
    fr = _load()
assert fr.__version__ == EXPECTED, "Wrong floodresq.py: expected version %s, found %s" % (EXPECTED, fr.__version__)
fr.setup_logging("floodresq.log")
print("floodresq version", fr.__version__)
print(fr.environment_info())

## 2. Settings
`BBOX` = (west, south, east, north). Check it on a map first. Make it as small as sensible: smaller area = faster, cheaper, fewer false alarms. For narrow Himalayan rivers a smaller `res_m` (for example 20) is better, but the grid may not exceed 2500 pixels per side.
`params` lets you override any value of `fr.DEFAULT_PARAMS`; whatever you set is saved in `run_config.json` and `results.json`. Do not tune thresholds by looking at EMSR927.

In [ ]:
from datetime import date, timedelta
import json
import numpy as np
import pandas as pd

CONFIG = {
    "event_date": "2026-08-26",
    "bbox": [85.10, 27.85, 85.45, 28.30],     # verify on a map!
    "osm_snapshot": "2026-07-27",             # must be BEFORE the event
    "res_m": 30,                              # pixel size in metres
    "search_days": 36,                        # search window before and after the event
    "pair_index": None,                       # None = automatic; or 0, 1, 2 to pick a row of the pairs table
    "scene_pad_s": 60,                        # seconds around the scene start used to select the radar data
    "params": {},                             # e.g. {"corridor_m": 500}
}
EVENT_DATE = date.fromisoformat(CONFIG["event_date"])
BBOX = tuple(CONFIG["bbox"])
OSM_SNAPSHOT = CONFIG["osm_snapshot"]
assert date.fromisoformat(OSM_SNAPSHOT) < EVENT_DATE, "OSM snapshot must be before the event"
grid = fr.make_grid(BBOX, CONFIG["res_m"])
PARAMS = dict(fr.DEFAULT_PARAMS)
PARAMS.update(CONFIG["params"])
with open("run_config.json", "w") as f:
    json.dump({"config": CONFIG, "params": PARAMS, "environment": fr.environment_info()}, f, indent=2, default=str)
print("Grid:", grid.shape, "pixels; each pixel is about %.0f x %.0f m" % (grid.dx_m, grid.dy_m))

## 3. Log in to Copernicus Data Space
Needs a free account and an **OAuth client** (Copernicus dashboard, User settings, OAuth clients). The client secret is shown only once when you create it. Type the id and secret when asked; they are hidden and never saved in the notebook. Do not share screenshots of them. `HTTP 401 invalid_client` means the id or secret is wrong: create a new client.

In [ ]:
import getpass
client = fr.CDSEClient(getpass.getpass("OAuth client id: "), getpass.getpass("OAuth client secret: "))
client.token()
print("OK: logged in")

## 4. Find a same-track before/after radar pair (search only)

In [ ]:
start = (EVENT_DATE - timedelta(days=CONFIG["search_days"])).isoformat()
end = (EVENT_DATE + timedelta(days=CONFIG["search_days"])).isoformat()
products = fr.search_s1(BBOX, start, end)
scenes = fr.scenes_table(products, BBOX)
pairs = fr.find_pairs(scenes, EVENT_DATE)
print(len(scenes), "scenes;", len(pairs), "same-track, same-direction pairs")
pair = fr.pick_pair(pairs, CONFIG["pair_index"])
control = fr.find_control(scenes, pair)
pair_warnings = fr.check_pair(pair, EVENT_DATE)      # raises if the pair does not straddle the event
print()
print("USING pair  : track", pair["rel_orbit"], pair["direction"], "| before", pair["pre_date"], "| after", pair["post_date"], "| gap", pair["gap_days"], "days")
print("Control scene:", None if control is None else str(control["date"]))
for w in pair_warnings:
    print("WARNING:", w)
pairs

## 5. Download the radar images (cropped to your area) and check them
Radar values are terrain-flattened backscatter (gamma0, linear power). Both VV and VH are kept. If available, the layover/shadow mask and local incidence angle are downloaded too and used to ignore unreliable pixels.

In [ ]:
pad = CONFIG["scene_pad_s"]
pre = fr.fetch_s1_scene(client, grid, pair["pre_scene"], pair["direction"], pad_s=pad)
post = fr.fetch_s1_scene(client, grid, pair["post_scene"], pair["direction"], pad_s=pad)
qc_pre, qc_post = fr.check_radar(pre, params=PARAMS), fr.check_radar(post, params=PARAMS)
print("pre :", qc_pre)
print("post:", qc_post)
ctrl_radar = None
if control is not None:
    ctrl_radar = fr.fetch_s1_scene(client, grid, control["name"], pair["direction"], pad_s=pad)
    print("ctrl:", fr.check_radar(ctrl_radar, params=PARAMS))
print("Layover/shadow bands used:", pre["geometry_available"] and post["geometry_available"])

### 5b. Terrain (DEM) - public Copernicus DEM, no special permission needed
The Sentinel Hub `dem` request returned `HTTP 403 COMMON_INSUFFICIENT_PERMISSIONS` on free accounts. This cell instead downloads the public Copernicus **GLO-30** tiles (30 m; GLO-90 as a fallback) from AWS Open Data and resamples them onto the same grid as the radar. Resampling only interpolates: **it never adds terrain detail finer than the source**, and the real source resolution is recorded in the results.

In [ ]:
dem, dem_info = fr.fetch_dem_public(grid, cache_dir="dem_cache")
print(dem_info)

In [ ]:
# CHECK 2
assert pre["vv"].shape == post["vv"].shape == dem.shape == grid.shape
dem_facts = fr.check_dem(dem, grid)
print("CHECK 2 OK:", dem_facts)
if dem_info["native_resolution_m"] and dem_info["native_resolution_m"] > (grid.dx_m * grid.dy_m) ** 0.5 * 1.5:
    print("NOTE: the DEM is coarser (%s m) than the radar pixels; slopes are smoothed." % dem_info["native_resolution_m"])

## 6. Optional: Sentinel-2 optical images (supporting evidence only)
Optical images are often cloudy in the monsoon. If clear enough, they are used as an independent check on the radar water; otherwise they are reported as not usable. Skip this cell if it errors.

In [ ]:
s2_pre = s2_post = None
try:
    t_a = (EVENT_DATE - timedelta(days=10)).isoformat() + "T00:00:00Z"
    t_b = EVENT_DATE.isoformat() + "T00:00:00Z"
    t_c = (EVENT_DATE + timedelta(days=12)).isoformat() + "T23:59:59Z"
    s2_pre = fr.fetch_s2_rgb(client, grid, t_a, t_b)
    s2_post = fr.fetch_s2_rgb(client, grid, t_b, t_c)
    print("Cloud fraction before: %.0f%%, after: %.0f%%" % (100 * s2_pre["cloud_fraction"], 100 * s2_post["cloud_fraction"]))
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 7))
    ax[0].imshow(s2_pre["rgb"]); ax[0].set_title("Sentinel-2 before"); ax[1].imshow(s2_post["rgb"]); ax[1].set_title("Sentinel-2 after")
    for a in ax: a.axis("off")
    plt.show()
except Exception as ex:
    s2_pre = s2_post = None
    print("Sentinel-2 step skipped:", ex)

## 7. Pre-event OpenStreetMap (buildings, roads, bridges, places, hospitals)
This can take 10 to 30 minutes on a busy server. Progress lines appear as pieces finish. Finished pieces are saved in `osm_cache`, so if the cell stops you can simply run it again and it continues. It only accepts data that the server states is **not newer than your snapshot date**.

In [ ]:
osm = fr.fetch_osm(BBOX, OSM_SNAPSHOT)
print({k: len(v) for k, v in osm.items() if k != "meta"})
print(osm["meta"])
targets = fr.make_targets(osm)
print("Hospitals:", len(osm["hospitals"]), "| towns/cities:", len(targets) - len(osm["hospitals"]))

In [ ]:
# CHECK 3
assert len(osm["rivers"]) > 0, "No rivers mapped in the box: widen BBOX"
assert len(osm["roads"]) > 0, "No roads found"
assert len(osm["places"]) > 0, "No named places found"
assert len(targets) > 0, "No hospital/town target: enlarge BBOX or add one by hand to `targets`"
print("CHECK 3 OK")
print("Rivers:", sorted({w["tags"].get("name", "?") for w in osm["rivers"]}))

## 8. Detect flood water and surface change from radar
What the detector does (all thresholds are in `fr.DEFAULT_PARAMS` and are saved in the results):
- **Speckle** is reduced with an edge-preserving Lee filter; a scene-wide brightness offset is measured far from the rivers and removed.
- **Open water**: dark in VV and VH after the event, not already dark before, a clear drop, gentle slope, low height above the nearest mapped river, and connected pixels only.
- **Surface change (candidate debris, sediment, wet ground)**: VV and VH change in the same direction by more than the noise (measured from the control pair), outside built-up areas and outside pixels that were unstable even without a flood.
- **Control**: the same rules on two scenes both BEFORE the event. Whatever it flags is noise.

In [ ]:
dist, rmask = fr.river_distance_m(grid, [w["coords"] for w in osm["rivers"]])
ctx = fr.terrain_context(grid, dem, dist, rmask, osm["buildings"], PARAMS)
det = fr.detect_flood(pre, post, ctx, ctrl_radar, PARAMS)
det_raw = fr.detect_flood(pre, post, ctx, ctrl_radar, PARAMS, use_unstable=False)
det_ctrl = fr.detect_flood(ctrl_radar, pre, ctx, None, PARAMS) if ctrl_radar is not None else None

A = lambda m: fr.area_km2(m, grid)
near_c = dist <= PARAMS["corridor_m"]
flood = {
    "water_km2": A(det["water"]),
    "change_increase_km2": A(det["change_increase"]), "change_decrease_km2": A(det["change_decrease"]),
    "permanent_water_km2": A(det["permanent_water"]),
    "builtup_km2": A(det["builtup"] & near_c), "unstable_km2": A(det["unstable"] & near_c & ~det["builtup"]),
    "control_water_km2": None if det_ctrl is None else A(det_ctrl["water"]),
    "control_change_km2": None if det_ctrl is None else A(det_ctrl["change_increase"] | det_ctrl["change_decrease"]),
}
print({k: (None if v is None else round(v, 3)) for k, v in flood.items()})
print("Noise: theory %.2f dB, measured from control %s dB; change threshold %.2f dB; brightness offset removed %.2f dB" % (
    det["stats"]["sigma_theory_db"], det["stats"]["sigma_empirical_db"], det["stats"]["change_threshold_db"], det["stats"]["bias_vv_db"]))
print("Valid radar share of the river corridor: %.0f%%" % (100 * det["stats"]["corridor_valid_fraction"]))

### 8b. Why is the water mask what it is? (diagnosis)
This table counts how many river-corridor pixels survive each test, one after the other. The first test that drops the count to zero is the reason for an empty water mask.

In [ ]:
import matplotlib.pyplot as plt
funnel_df = pd.DataFrame(fr.funnel(det))
display(funnel_df)
if det["water"].sum() == 0:
    for line in fr.explain_empty_water(det, grid):
        print("-", line)
json.dump(funnel_df.to_dict("records"), open("diagnostics_funnel.json", "w"), indent=2)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
vals = det["post_vv_db"][det["valid"] & (dist <= PARAMS["water_corridor_m"])]
ax[0].hist(vals[np.isfinite(vals)], bins=80); ax[0].axvline(PARAMS["water_vv_db"], color="r"); ax[0].set_title("VV after (dB), river corridor; red = water threshold")
d = det["diff_vv_db"][det["valid"] & (dist <= PARAMS["water_corridor_m"])]
ax[1].hist(d[np.isfinite(d)], bins=80); ax[1].set_title("VV change after minus before (dB), river corridor")
plt.show()

### Check EVERY radar pair for water (diagnosis, not tuning)
This runs the same water detector on all before/after pairs and shows, per pair, how many candidate pixels survive each test and where they are lost. It downloads two extra scenes per pair (uses your free quota). Do not change thresholds to match the EMSR927 maps; if you change any threshold because of what you see here, write it in `CONFIG["params"]` and say so in the report.


In [ ]:
scan = fr.pair_water_scan(client, grid, pairs, ctx, PARAMS, pad_s=CONFIG["scene_pad_s"])
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
display(scan)
scan.to_csv("pair_water_scan.csv", index=False)


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 7))
ax[0].imshow(fr.to_db(pre["vv"]), cmap="gray", vmin=-25, vmax=0); ax[0].set_title("VV before (dB) " + str(pair["pre_date"]))
ax[1].imshow(fr.to_db(post["vv"]), cmap="gray", vmin=-25, vmax=0); ax[1].set_title("VV after (dB) " + str(pair["post_date"]))
ax[2].imshow(fr.to_db(post["vv"]), cmap="gray", vmin=-25, vmax=0)
ov = np.zeros(grid.shape + (4,))
ov[det["builtup"] & near_c] = (0.6, 0.6, 0.6, 0.35)
ov[det["change_increase"]] = (1, 0.55, 0, 0.8); ov[det["change_decrease"]] = (0.55, 0.27, 0.68, 0.8); ov[det["water"]] = (0, 0.4, 1, 0.9)
ax[2].imshow(ov); ax[2].set_title("Blue = water, orange = increase, purple = decrease, grey = not assessed")
for a in ax: a.axis("off")
plt.show()

In [ ]:
# CHECK 4: is the detection plausible, or obviously broken? Zero detections are allowed: the notebook never forces a result.
assert not (det["water"] & det["change_increase"]).any()
n_corr = int((dist <= PARAMS["water_corridor_m"]).sum())
flagged = int(det["water"].sum() + det["change_increase"].sum() + det["change_decrease"].sum())
if flagged > 0.5 * n_corr:
    print("WARNING: more than half of the river corridor is flagged: almost certainly noise or a wrong pair.")
if flagged == 0:
    print("Nothing flagged. That is a valid outcome; see the diagnosis table above for why.")
print("CHECK 4 done (this only means 'not obviously broken', not 'accurate')")

## 9. How much evidence is there?
Combines coverage, noise, the control comparison and (if usable) Sentinel-2. The answer is one of: **water_detected**, **no_water_detected**, **inconclusive**. Confidence is at most 'medium' because nothing is validated against reference data.

In [ ]:
s2 = None
if s2_pre is not None and s2_post is not None:
    s2 = fr.s2_support(s2_pre, s2_post, det["water"], dist <= PARAMS["water_corridor_m"], grid)
    print("Sentinel-2 support:", s2)
qc = {"corridor_valid_fraction": det["stats"]["corridor_valid_fraction"],
      "sigma_empirical_db": det["stats"]["sigma_empirical_db"],
      "geometry_bands": bool(qc_pre["geometry_bands"] and qc_post["geometry_bands"])}
evidence = fr.assess_evidence(flood, qc, PARAMS, s2, pair_warnings)
print("VERDICT:", evidence["verdict"], "| confidence:", evidence["confidence"])
for r in evidence["reasons"]:
    print("  -", fr._reason_text(r, "en"))

## 10. Potentially affected assets and who might be cut off
Two hazard definitions are used, so you can see how sensitive the answer is: **likely** (suspected open water only) and **possible** (any flagged pixel). Assets are only ever 'potentially affected'.

In [ ]:
G = fr.build_road_graph(osm["roads"])
change_all = det["change_increase"] | det["change_decrease"]
access = fr.access_scenarios(G, grid, {"likely": det["water"], "possible": det["water"] | change_all}, osm["places"], targets)
bld = fr.building_damage(grid, osm["buildings"], det["water"], change_all)
bridges = fr.bridge_damage(grid, osm["bridges"], det["water"], change_all)
baseline = fr.baseline_components(G, targets)
if evidence["verdict"] == "inconclusive":
    print("NOTE: evidence is inconclusive, so the numbers below are conditional on the flagged pixels being real.")
for name in ("likely", "possible"):
    rows = access[name]["rows"]
    print(name.upper(), "| roads:", access[name]["roads"], "|", pd.Series([r["status"] for r in rows]).value_counts().to_dict())
print("Buildings:", {k: v for k, v in bld.items() if k != "hit_points"})
print("Bridges: total", bridges["total"], "intersecting flagged pixels", bridges["possibly_hit"])
print("Road network before the event:", baseline)
pd.DataFrame(access["possible"]["rows"])[["name", "name_en", "status", "dist_pre_km", "dist_post_km"]].sort_values("status").head(40)

## 11. Bonus: trace the flood path downhill from a point
Set `UPSTREAM_LON/LAT` to a point inside your area. This is a very simple steepest-descent trace on a coarse DEM, so it can stop early in pits; it prints why it stopped.

In [ ]:
UPSTREAM_LON, UPSTREAM_LAT = None, None     # e.g. a point you chose inside the BBOX
trace_info = None
if UPSTREAM_LON is not None:
    path = fr.trace_downstream(dem, grid, UPSTREAM_LON, UPSTREAM_LAT)
    near = fr.settlements_near_path(path, osm["places"], 500)
    trace_info = {"stopped_because": path["stopped_because"], "settlements": [n["name_en"] or n["name"] for n in near]}
    print(trace_info)
else:
    print("Skipped (set UPSTREAM_LON / UPSTREAM_LAT to run).")

## 12. Save results.json (the only source of numbers) and freeze the run

In [ ]:
results = fr.build_results(grid, EVENT_DATE, pair, control, det, det_ctrl, det_raw, flood, qc, dem_info, osm["meta"],
                          evidence, bld, bridges, access, baseline, config={"config": CONFIG, "params": PARAMS}, s2=s2, trace=trace_info)
with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)
for w in fr.quality_warnings(results):
    print("WARNING:", w)
frozen_masks = {"water": det["water"], "change_increase": det["change_increase"], "change_decrease": det["change_decrease"]}
frozen = fr.freeze_run(results, frozen_masks)
print("Saved results.json; fingerprint", frozen["sha256"][:16], "...")
print(json.dumps({k: results[k] for k in ("evidence", "flood", "damage", "access")}, indent=2, ensure_ascii=False)[:3500])

## 13. The situation-report copilot (English and Nepali)
Reports are built only from `results.json`. If you add an LLM key, the model's answer is checked: any number that is not in `results.json`, any wording that states damage or isolation as fact, or an answer with no uncertainty wording is **rejected** and replaced by the template answer. Without a key you still get a working template-based copilot. Every answer ends with sources and timestamps. Have a Nepali speaker review the Nepali wording before any demo.

In [ ]:
print(fr.report_text(results, "en"))
print("\n" + "=" * 70 + "\n")
print(fr.report_text(results, "ne"))

In [ ]:
# Optional LLM. Leave the key blank (just press Enter) to use templates only.
import getpass
API_KEY = getpass.getpass("Anthropic API key (blank = no LLM): ").strip() or None
MODEL = "claude-sonnet-5-5"
for q, lang in [("Which villages might be cut off?", "en"), ("कति भवन प्रभावित हुन सक्छन्?", "ne"), ("How reliable is this?", "en")]:
    out = fr.ask_copilot(q, results, lang, api_key=API_KEY, model=MODEL)
    print("Q:", q, "| source:", out["source"], "| rejected:", out["rejected"])
    print(out["answer"], "\n")

## 14. Map dashboard and one-page report
Dashboard layers: radar before/after (switch on in the layer panel), suspected water, surface-change candidates (increase and decrease), pre-existing water, areas not assessed, roads, flagged road segments, bridges, buildings, settlements by status, pre-event routes of possibly cut-off settlements, hospitals/towns. Legend, dates, sources, uncertainty and limits are in the two boxes.

In [ ]:
masks = {"water": det["water"], "change_increase": det["change_increase"], "change_decrease": det["change_decrease"],
         "permanent_water": det["permanent_water"], "not_assessed": (det["builtup"] | det["unstable"]) & near_c}
impacts = {"blocked_possible": access["possible"]["blocked"], "blocked_likely": access["likely"]["blocked"],
           "bridge_rows": bridges["rows"], "bld": bld}
imagery = {
    "Radar VV before (%s)" % pair["pre_date"]: {"array": fr.to_db(np.where(fr.reliable_mask(pre), pre["vv"], np.nan)), "kind": "gray", "vmin": -25, "vmax": 0},
    "Radar VV after (%s)" % pair["post_date"]: {"array": fr.to_db(np.where(fr.reliable_mask(post), post["vv"], np.nan)), "kind": "gray", "vmin": -25, "vmax": 0},
}
if s2 is not None and s2["usable"]:
    imagery["Sentinel-2 before (cloud %.0f%%)" % (100 * s2_pre["cloud_fraction"])] = {"array": s2_pre["rgb"], "kind": "rgb"}
    imagery["Sentinel-2 after (cloud %.0f%%)" % (100 * s2_post["cloud_fraction"])] = {"array": s2_post["rgb"], "kind": "rgb"}
routes = fr.routes_to_nearest_target(G, targets, access["possible"]["rows"])
fr.make_dashboard("dashboard.html", grid, masks, osm, impacts, access, targets, results, imagery=imagery, routes=routes)
fr.make_report_html("situation_report.html", results)
print("Created dashboard.html and situation_report.html. Download them from the Files panel (the map does not show inside Colab).")

To get the PDF: download `situation_report.html`, open it in **Chrome**, press Ctrl+P, choose **Save as PDF**, and check the preview shows ONE page (shrink 'Scale' if not). Open `dashboard.html` in Chrome too, after extracting it from the ZIP (it does not work from inside a zip).

## 15. EVALUATION ONLY: compare with the official EMSR927 maps
Run this **last** and only once. Download the EMSR927 flood-extent vector file (GeoJSON/shapefile/zip) yourself from mapping.emergency.copernicus.eu, upload it, and set the path. Also upload the product's area-of-interest layer if there is one, because only that area is scored. The run must still match the frozen fingerprint, otherwise the evaluation refuses to run.
Never change thresholds after seeing this score. Report it as-is, including a bad score. Only radar **water** is scored as flood extent; surface-change candidates are not flood extent.

In [ ]:
REFERENCE_PATH = None      # e.g. "EMSR927_flood_extent.geojson"
AOI_PATH = None            # e.g. "EMSR927_aoi.geojson"
evaluation = None
if REFERENCE_PATH:
    evaluation = fr.evaluate_vs_reference(det["water"], grid, REFERENCE_PATH, AOI_PATH, results=results, masks=frozen_masks)
    print(evaluation)
    json.dump(evaluation, open("evaluation.json", "w"), indent=2)
else:
    print("Skipped: set REFERENCE_PATH first. No validation score exists for this run.")

## 16. Bundle everything for submission

In [ ]:
files_to_zip = ["results.json", "dashboard.html", "situation_report.html", "frozen_run.json", "run_config.json",
                "diagnostics_funnel.json", "pair_water_scan.csv", "floodresq.log", "floodresq.py"] + (["evaluation.json"] if os.path.exists("evaluation.json") else [])
missing = fr.bundle_outputs("floodresq_outputs.zip", files_to_zip)
print("Missing files:", missing if missing else "none")
from google.colab import files
files.download("floodresq_outputs.zip")